# Generate Examples for the ML4H Poster

In [ ]:
from ultrabench2 import load_dataset
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from matplotlib.colors import ListedColormap
from PIL import Image

EXAMPLE_IDXS = {
    "AUL": 1,
    "BUS-BRA": 7,
    "CAMUS": 7,
    "COVID-BLUES": 1,
    "FetalPlanes": 0,
    "GBCU": 0,
    "MASLD": 0,
    "MMOTU": 18,
    "OpenKidney": 1,
    "PSFHS": 7,
    "STMUS": 0,
    "TN3K": 0,
}

OUTPUT_DIR = Path("examples")
OUTPUT_DIR.mkdir(exist_ok=True)

OUTPUT_SIZE = 1200
FIGURE_SIZE_IN = 4
DPI = 300
MASK_ALPHA = 0.5

# Local datasets need the path to the raw data.
RAW_PATHS = {
    "GBCU": "/project/data/raw/GBCU-Shared",
    "OpenKidney": "/project/data/raw/kidneyUS",
}

# Subset to load for datasets that have more than one.
SUBSETS = {
    "CAMUS": "segmentation",
    "FetalPlanes": "fetal",
}

# The mask to overlay, or None for classification-only datasets.
MASK_COLUMNS = {
    "AUL": "mass_mask",
    "BUS-BRA": "tumor_mask",
    "CAMUS": "mask",
    "COVID-BLUES": None,
    "FetalPlanes": None,
    "GBCU": None,
    "MASLD": None,
    "MMOTU": "tumor_mask",
    "OpenKidney": "regions_mask",
    "PSFHS": "psfh_mask",
    "STMUS": "mask",
    "TN3K": "nodule_mask",
}

# One colour per foreground class (class 1 uses the first colour, etc.).
CLASS_COLOURS = ["#e6194b", "#3cb44b", "#4363d8", "#ffe119"]

In [ ]:
def pad_and_resize(image, size, resample):
    """Pad an image to a square (centred) and resize it to size x size."""
    width, height = image.size
    side = max(width, height)
    padded = Image.new(image.mode, (side, side), 0)
    padded.paste(image, ((side - width) // 2, (side - height) // 2))

    return padded.resize((size, size), resample=resample)


for name, idx in EXAMPLE_IDXS.items():
    example = load_dataset(
        name=name,
        path=RAW_PATHS.get(name),
        subset=SUBSETS.get(name),
        split="test",
    )[idx]

    image = pad_and_resize(
        example["image"].convert("RGB"), OUTPUT_SIZE, Image.Resampling.BICUBIC
    )

    fig = plt.figure(figsize=(FIGURE_SIZE_IN, FIGURE_SIZE_IN), dpi=DPI)
    ax = fig.add_axes([0, 0, 1, 1])
    ax.imshow(image)

    mask_column = MASK_COLUMNS[name]
    if mask_column is not None:
        # Nearest-neighbour keeps the class labels intact.
        mask = np.array(
            pad_and_resize(
                example[mask_column].convert("L"),
                OUTPUT_SIZE,
                Image.Resampling.NEAREST,
            )
        )
        masked = np.ma.masked_equal(mask, 0)
        cmap = ListedColormap(CLASS_COLOURS)
        ax.imshow(
            masked,
            cmap=cmap,
            vmin=1,
            vmax=len(CLASS_COLOURS),
            alpha=MASK_ALPHA,
            interpolation="nearest",
        )

    ax.axis("off")
    fig.savefig(OUTPUT_DIR / f"{name}.png", dpi=DPI)
    plt.close(fig)
    print(f"Saved {OUTPUT_DIR / f'{name}.png'}")